In [1]:
import json
import math
import h5py
import numpy as np
import tensorflow as tf

from keras.layers import (
    Input,
    Dense,
    Embedding,
    LSTM,
    Dropout,
    RepeatVector,
    Concatenate,
    TimeDistributed,
)

from keras.models import Model

# =====================================================
# CONFIG
# =====================================================

with open("../config/config.json") as f:
    config = json.load(f)

MAX_LEN = config["max_len"]
VOCAB_SIZE = config["vocab_size"]

# =====================================================
# CONSTANTS
# =====================================================

FEATURE_DIM = 512
EMBED_DIM = 256

BATCH_SIZE = 64
EPOCHS = 30

TRAIN_RECORD = "../data/records/train_25426.tfrecord"
VAL_RECORD = "../data/records/val_3178.tfrecord"

FEATURES_FILE = "../data/images/flickr30k_vgg16_features.h5"

BEST_MODEL_PATH = "../output/best_model.keras"
FINAL_MODEL_PATH = "../output/final_model.keras"
HISTORY_PATH = "../output/history.json"

2026-06-29 22:05:15.632407: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def define_model(vocab_size, max_length):

    image_input = Input(
        shape=(FEATURE_DIM,),
        name="image_features",
    )

    image = Dropout(0.5)(image_input)

    image = Dense(
        EMBED_DIM,
        activation="relu",
        name="image_projection",
    )(image)

    h0 = Dense(
        EMBED_DIM,
        activation="tanh",
        name="initial_hidden",
    )(image)

    c0 = Dense(
        EMBED_DIM,
        activation="tanh",
        name="initial_cell",
    )(image)

    caption_input = Input(
        shape=(max_length,),
        name="caption",
    )

    x = Embedding(
        input_dim=vocab_size,
        output_dim=EMBED_DIM,
        mask_zero=True,
    )(caption_input)

    x = Dropout(0.5)(x)

    x = LSTM(
        EMBED_DIM,
        return_sequences=True,
        dropout=0.3,
    )(x, initial_state=[h0, c0])

    image_sequence = RepeatVector(max_length)(image)

    x = Concatenate()([x, image_sequence])

    x = TimeDistributed(
        Dense(EMBED_DIM, activation="relu")
    )(x)

    x = Dropout(0.5)(x)

    outputs = TimeDistributed(
        Dense(vocab_size, activation="softmax")
    )(x)

    model = Model(
        inputs=[image_input, caption_input],
        outputs=outputs,
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3), # type: ignore
        loss="sparse_categorical_crossentropy",
    )

    return model

In [3]:
features = h5py.File(FEATURES_FILE, "r")


def count_examples(path):
    return sum(1 for _ in tf.data.TFRecordDataset(path))


train_size = count_examples(TRAIN_RECORD)
val_size = count_examples(VAL_RECORD)

print(f"Training examples   : {train_size}")
print(f"Validation examples : {val_size}")


def parse_example(proto):

    feature_desc = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "input": tf.io.FixedLenFeature([MAX_LEN], tf.int64),
        "target": tf.io.FixedLenFeature([MAX_LEN], tf.int64),
    }

    return tf.io.parse_single_example(proto, feature_desc)


def feature_lookup(image_id):

    image_id = image_id.numpy().decode()

    return features[image_id][:].astype(np.float32) # type: ignore


def load_features(example):

    feature = tf.py_function(
        feature_lookup,
        [example["image"]],
        tf.float32,
    )

    feature.set_shape((FEATURE_DIM,)) # type: ignore

    example["image"] = feature

    return example


def prepare(example):

    return (
        {
            "image_features": example["image"],
            "caption": example["input"],
        },
        example["target"],
    )


def build_dataset(
    path,
    shuffle=False,
):

    dataset = tf.data.TFRecordDataset(path)

    dataset = dataset.map(
        parse_example,
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    dataset = dataset.map(
        load_features,
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    dataset = dataset.map(
        prepare,
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    if shuffle:
        dataset = dataset.shuffle(
            min(train_size, 10000),
            reshuffle_each_iteration=True,
        )

    dataset = dataset.repeat()

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_set = build_dataset(
    TRAIN_RECORD,
    shuffle=True,
)

val_set = build_dataset(
    VAL_RECORD,
)

2026-06-29 22:05:30.256468: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Training examples   : 127130
Validation examples : 15890


2026-06-29 22:05:31.699275: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [4]:
steps_per_epoch = math.ceil(train_size / BATCH_SIZE)
validation_steps = math.ceil(val_size / BATCH_SIZE)

print(f"Steps per epoch : {steps_per_epoch}")
print(f"Validation steps: {validation_steps}")

model = define_model(
    VOCAB_SIZE,
    MAX_LEN,
)

callbacks = [
    tf.keras.callbacks.EarlyStopping( # type: ignore
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ), 
    tf.keras.callbacks.ReduceLROnPlateau( # type: ignore
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint( # type: ignore
        BEST_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    train_set,
    validation_data=val_set,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
)

Steps per epoch : 1987
Validation steps: 249
Epoch 1/30


/Users/cocco/Desktop/flickr30k/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1987/1987 ━━━━━━━━━━━━━━━━━━━━ 0s 685ms/step - loss: 2.6371
Epoch 1: val_loss improved from None to 2.28834, saving model to ../output/best_model.keras

Epoch 1: finished saving model to ../output/best_model.keras
1987/1987 ━━━━━━━━━━━━━━━━━━━━ 1421s 711ms/step - loss: 2.6371 - val_loss: 2.2883 - learning_rate: 0.0010
Epoch 2/30
1987/1987 ━━━━━━━━━━━━━━━━━━━━ 0s 732ms/step - loss: 2.2906
Epoch 2: val_loss improved from 2.28834 to 2.17037, saving model to ../output/best_model.keras

Epoch 2: finished saving model to ../output/best_model.keras
1987/1987 ━━━━━━━━━━━━━━━━━━━━ 1522s 766ms/step - loss: 2.2906 - val_loss: 2.1704 - learning_rate: 0.0010
Epoch 3/30
1987/1987 ━━━━━━━━━━━━━━━━━━━━ 0s 864ms/step - loss: 2.1911
Epoch 3: val_loss improved from 2.17037 to 2.11165, saving model to ../output/best_model.keras

Epoch 3: finished saving model to ../output/best_model.keras
1987/1987 ━━━━━━━━━━━━━━━━━━━━ 1790s 901ms/step - loss: 2.1911 - val_loss: 2.1117 - learning_rate: 0.0010
Epoch 4/30
1

In [5]:
model.save(FINAL_MODEL_PATH)

with open(HISTORY_PATH, "w") as f:
    json.dump(
        history.history,
        f,
        indent=4,
    )

features.close()

print("Training completato.")
print(f"Best model  : {BEST_MODEL_PATH}")
print(f"Final model : {FINAL_MODEL_PATH}")
print(f"History     : {HISTORY_PATH}")

Training completato.
Best model  : ../output/best_model.keras
Final model : ../output/final_model.keras
History     : ../output/history.json
